# Parquet con pandas y PySpark

¿Cuántas unidades se han pedido en cada zona? Usaremos el Parquet de un millón de pedidos que ya guardamos en el [notebook 05](05_csv_vs_parquet_pyspark.ipynb).

Ejecuta primero el 05 hasta guardar el Parquet y abre ambos notebooks desde `PySpark/notebooks`. Si te faltan pandas o PyArrow, ejecuta `%pip install pandas pyarrow` en una celda del mismo notebook.

PyArrow es la biblioteca que pandas usa para leer y escribir archivos Parquet. ``engine="pyarrow"`` le indica a pandas que le encargue ese trabajo.

## 1. Leer solo `zona` y `unidades`

Ambos leen la misma carpeta Parquet del 05. 
- Pandas carga las dos columnas seleccionadas en la memoria de Python. 
- PySpark prepara la lectura y puede consultar metadatos al abrir el Parquet. Leerá las filas cuando le pidamos un resultado.

In [ ]:
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, functions as F

ruta_parquet = Path("../data/pyspark_pedidos/pedidos_parquet").resolve()


In [ ]:

datos_pandas = pd.read_parquet(ruta_parquet, columns=["zona", "unidades"], engine="pyarrow")

spark = SparkSession.builder.master("local[2]").appName("Practica Parquet").getOrCreate()
datos_spark = spark.read.parquet(str(ruta_parquet)).select("zona", "unidades")

## 2. Agrupar por zona y sumar las unidades

En pandas, el total se calcula al llamar a `sum()`. En Spark, la agregación forma parte de un plan. En `explain()`, busca `ReadSchema` y comprueba qué columnas leerá del Parquet.

In [ ]:
total_pandas = datos_pandas.groupby("zona", as_index=False)[["unidades"]].sum()

total_spark = datos_spark.groupBy("zona").agg(F.sum("unidades").alias("unidades"))
total_spark.explain()

## 3. Comparar los resultados

Muestra los dos resúmenes ordenados por zona y comprueba que coinciden los totales de las 20 zonas.

In [ ]:
total_pandas.sort_values("zona").head(10)

En PySpark, `orderBy()` devuelve un DataFrame con el orden definido en su plan. `show()` ejecuta el cálculo y muestra las 20 filas del resumen como una tabla de texto.

In [ ]:
total_spark.orderBy("zona").show(10)

In [ ]:
spark.stop()

## Navegación

[Anterior · CSV vs. Parquet](05_csv_vs_parquet_pyspark.ipynb) · [Volver al README](../README.md) · [Siguiente · Spark y HDFS](07_spark_hdfs.ipynb)